In [22]:
from keras.models import Model 
from keras.layers import Input, Convolution2D, MaxPooling2D, Dense, Dropout, Flatten, Dense, Dropout, Flatten, Conv2D, MaxPooling2D
# import np_utils
import numpy as np
import pandas as pd
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import os
import datetime
from sklearn.model_selection import train_test_split

from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import SpectralClustering


from sklearn.utils import resample
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import pairwise_distances_argmin_min
from tslearn.metrics import cdist_dtw
from sklearn.metrics import silhouette_score
from tslearn.clustering import silhouette_score

import tensorflow as tf
import csv, json
from pathlib import Path

In [23]:
from print_versions import print_versions
import sys
import sklearn
print_versions(globals())
print(sklearn.__version__)
print(sys. version)

keras==3.9.2
numpy==1.26.4
pandas==2.2.2
tslearn==0.6.3
tensorflow==2.19.0
csv==1.0
json==2.0.9
sklearn==1.5.1
1.5.1
3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]


In [24]:
RANDOM_STATE = 42
sil_sample=20000

# Define functions

In [25]:
def agglom_dtw(embeds, k, distance):

    embeds_3D = np.expand_dims(embeds, axis=-1)
    sample_size = 1000  
    sampled_embeds = resample(embeds_3D, n_samples=sample_size, random_state=42)

    dtw_distance_matrix = cdist_dtw(sampled_embeds)

    hc = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    sampled_labels = hc.fit_predict(dtw_distance_matrix)

 
    cluster_centroids = []
    for i in range(k):
        cluster_indices = np.where(sampled_labels == i)[0]
        cluster_series = sampled_embeds[cluster_indices]
        dist_matrix_cluster = cdist_dtw(cluster_series)
        medoid_idx_in_cluster = np.argmin(dist_matrix_cluster.sum(axis=1))
        cluster_centroids.append(cluster_series[medoid_idx_in_cluster])
        
    cluster_centroids = np.array(cluster_centroids)  
    
    from tslearn.clustering import TimeSeriesKMeans

    ts_kmeans = TimeSeriesKMeans(
        n_clusters=k,
        metric="dtw",
        max_iter=0,
        random_state=42
    )
    ts_kmeans.cluster_centers_ = cluster_centroids

    labels = ts_kmeans.predict(embeds_3D)
    
    return labels


In [26]:
def agglomerative(embeds, k, distance, assign_method):

    sample_size = 5000 
    sampled_embeds = resample(embeds, n_samples=sample_size, random_state=42)
    
    hierarchical_cluster = AgglomerativeClustering(n_clusters=k, metric=distance, linkage='ward')
    labels_sub = hierarchical_cluster.fit_predict(sampled_embeds)
    
    if (assign_method == 'knn'): 
        neigh = KNeighborsClassifier(n_neighbors=13)
        neigh.fit(sampled_embeds, labels_sub)
        labels = neigh.predict(embeds) 
    else:      
        centroids_model = NearestCentroid()
        centroids_model.fit(sampled_embeds, labels_sub)
        cluster_centroids = centroids_model.centroids_
        labels, _  = pairwise_distances_argmin_min(embeds, cluster_centroids)

    return labels

In [27]:
def sil_score(X_embed, labels, s, debug=1): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="euclidean")
    else:
        sil = silhouette_score(X_embed, labels, metric="euclidean")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [28]:
def sil_score_dtw(X_embed, labels, s, debug=1): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="dtw")
    else:
        sil = silhouette_score(X_embed, labels, metric="dtw")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [29]:
def DAE_reduction(df, bottleneck, ep, batch):
    df_flat = [sample.flatten() for sample in df]
    df_flat = np.asarray(df_flat)
    train, test = train_test_split(df_flat, test_size=0.20, random_state=42)
    input = Input(shape=(df_flat.shape[1],))

    encoded = Dense(30, activation='relu')(input)
    encoded = Dense(20, activation='relu')(encoded)
    encoded = Dense(10, activation='relu')(encoded)
    encoded = Dense(bottleneck, activation='linear', name='bottleneck')(encoded)

    decoded = Dense(bottleneck, activation='relu')(encoded)
    decoded = Dense(20, activation='relu')(decoded)
    decoded = Dense(30, activation='relu')(decoded)
    output = Dense(df_flat.shape[1], activation=None)(decoded)

    autoencoder = Model(input, output)
#   autoencoder.summary()
#   autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    huber = tf.keras.losses.Huber()
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss=huber)
    autoencoder.fit(train, train,
     epochs=ep,
     batch_size=batch,
     shuffle=True,
     validation_data=(test, test), verbose = 0)
    encoder = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer('bottleneck').output)
    encoded_ts = encoder.predict(df_flat)
    return encoded_ts, encoder 

In [30]:
# Hyperparameter tuning; 
# hardcoded for DAE with linear bottleneck layer, 2048 batch size and Huber loss; 
# may want to tune them as well

def hypertune(target, k, clustering, abm): 

    BOTTLENECK_MIN = 5 
    EPOCH_MIN = 3
    
    bottleneck = 5     #  number of desired embeddings; tune in the range of [BOTTLENECK_MIN..bottleneck]
    epochs = 3         #  tune in range of [EPOCH_MIN..epochs] 
    batch = 2048       #  fixed from prior trials 
    evals = 10           #  number of repeated trainings to reduce noise in model 
  
    sil_sample = 20000 # silhouette sample 

    max_sil_score = 0         # max silohouette score 
    best_embed = []           # best emebeddings 
    best_model = []           # model that produced the best embeddings 
    
    for b in np.arange(BOTTLENECK_MIN, bottleneck + 1):   # bottleneck 
        for e in np.arange(EPOCH_MIN, epochs + 1):        # epoch
            for i in np.arange(1, evals + 1):             # evals 
                
                print(f"Evaluation: embeddings {b}, epoch {e}, eval # {i}:")
                # get embeddings 
                dae_embeds, dae_model = DAE_reduction(raw_data[target], bottleneck, epochs, batch)

                # clustering 
                if (clustering == 'agglom-dtw'):
                 #   dae_labels = agglomerative(dae_embeds, k, 'euclidean', 'knn')
                    dae_labels = agglom_dtw(dae_embeds, k, 'dtw')
                elif (clustering == 'agglom-euc'):
                    dae_labels = agglomerative(dae_embeds, k, 'euclidean', 'knn')
                    #dae_labels = agglom_dtw(dae_embeds, k, 'dtw')
                else: 
                    dae_labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit(dae_embeds).labels_
                
                unique_labels = np.unique(dae_labels)
                if (len(unique_labels) > 1): 
                    # silhouette score 
                    score = sil_score(dae_embeds, dae_labels, sil_sample)
                    # save the best embeddings and model   
                    if (score > max_sil_score): 
                        max_sil_score = score
                        best_embed = dae_embeds
                        best_model.append(dae_model) # assuming we are hypertuning the three models in order  
                        best_params = str(b) + '_' + str(e) + '_' + str(batch) + '_linear'
                    # log result 
                    outfile = 'dae_sil_scores_all_dtw_' + clustering + '.csv'
                    with open(outfile, "a") as f:
                        f.write(f"{abm},{k},{b},{e},{i},{score:.4f}\n")
                else: 
                    print(f"Error: Too few labels generated. Skipping ...")
 
    print(f"Best Silhouette: {max_sil_score:.4f}")
    print(f"Best parameters: {best_params}")
    filename = 'extracted_features/' + str(target) + '_DAE_' + best_params + '_' + clustering + '.csv'
    np.savetxt(filename, best_embed, delimiter=",")
    with open("daep_sil_scores.csv", "a") as f:
        f.write(f"{abm},{k},{max_sil_score:.4f}\n")

In [31]:
def import_ff_data(filename):
    expected_columns=155
    data = []
    with open(filename, 'r') as file:
        for line in file:
            row = line.strip().split(',')
            if len(row) < expected_columns:
                row += [np.nan] * (expected_columns - len(row))
            data.append(row)
    df = pd.DataFrame(data)
    def fill_last_valid(row):
        for i in range(1, len(row)):
            if pd.isna(row[i]):
                row[i] = row[i - 1]  
        return row
    df_filled = df.apply(fill_last_valid, axis=1)
    return df_filled

In [32]:
def save_labels_ndjson(method: str, abm: str, k: int, labels, json_path="cluster_labels.ndjson"):
    rec = {
        "method": method,          # e.g., "raw", "pca", "dae", "dcae", "daep", "daep-spectral", "daep-agglom-knn", etc.
        "abm": abm,                # e.g., "BR", "ECV", "FF"
        "k": int(k),
        "labels": [int(x) for x in labels]  # ensure ints for consistency
    }
    with open(json_path, "a", encoding="utf-8") as jf:
        jf.write(json.dumps(rec))
        jf.write("\n")

# Import Data

In [33]:
poor = pd.read_csv("SimData/bank_reserves_outputs_poor.csv", header=None)
middle = pd.read_csv("SimData/bank_reserves_outputs_middle.csv", header=None)
rich = pd.read_csv("SimData/bank_reserves_outputs_rich.csv", header=None)
sc = StandardScaler()
br = []
for i in np.arange(0, poor.shape[0]):
    sample = pd.concat([poor.iloc[i], middle.iloc[i]], axis=0).T
    sample = pd.concat([sample, rich.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    br.append(sample_std)

In [34]:
ecv_active = pd.read_csv("SimData/epsteinCV_outputs_active.csv", header=None)
ecv_jailed = pd.read_csv("SimData/epsteinCV_outputs_jailed.csv", header=None)
ecv_quiet = pd.read_csv("SimData/epsteinCV_outputs_quiet.csv", header=None)
sc = StandardScaler()
ecv = []
for i in np.arange(0, ecv_active.shape[0]):
    sample = pd.concat([ecv_active.iloc[i], ecv_jailed.iloc[i]], axis=0).T
    sample = pd.concat([sample, ecv_quiet.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ecv.append(sample_std)

In [35]:
ff_onfire = import_ff_data("SimData/forest_fire_outputs_onfire.csv")
print("check 1")
ff_fine = import_ff_data("SimData/forest_fire_outputs_fine.csv")
print("check 2")
ff_burned = import_ff_data("SimData/forest_fire_outputs_burned.csv")
sc = StandardScaler()
ff = []
for i in np.arange(0, ff_onfire.shape[0]):
    sample = pd.concat([ff_onfire.iloc[i], ff_fine.iloc[i]], axis=0).T
    sample = pd.concat([sample, ff_burned.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ff.append(sample_std)

check 1
check 2


In [36]:
ABMs = ["BR", "ECV", "FF"]
raw_data = []
raw_data.append(br)
raw_data.append(ecv)
raw_data.append(ff)

In [37]:
# Extract PCA embeddings; not re-doing PCA in this notebook
pca_br_embeds = pd.read_csv('extracted_features/bank_reserves_pca_standardized.csv')
pca_br_embeds = pca_br_embeds.drop('Unnamed: 0', axis=1)
pca_br_embeds = pca_br_embeds.to_numpy()

pca_ecv_embeds = pd.read_csv('extracted_features/epstein_pca_standardized.csv')
pca_ecv_embeds = pca_ecv_embeds.drop('Unnamed: 0', axis=1)
pca_ecv_embeds = pca_ecv_embeds.to_numpy()

pca_ff_embeds = pd.read_csv('extracted_features/forestfire_pca_standardized.csv')
pca_ff_embeds = pca_ff_embeds.drop('Unnamed: 0', axis=1)
pca_ff_embeds = pca_ff_embeds.to_numpy()

pca_embeds = []
pca_embeds.append(pca_br_embeds)
pca_embeds.append(pca_ecv_embeds)
pca_embeds.append(pca_ff_embeds)

In [38]:
# Extract DAE embeddings; not re-doing DAE in this notebook
dae_br_embeds = pd.read_csv('extracted_features/bank_reserves_dae.csv', header=None)
#dae_br_embeds = dae_br_embeds.drop('Unnamed: 0', axis=1)
dae_br_embeds = dae_br_embeds.to_numpy()

dae_ecv_embeds = pd.read_csv('extracted_features/epstein_dae.csv', header=None)
#dae_ecv_embeds = dae_ecv_embeds.drop('Unnamed: 0', axis=1)
dae_ecv_embeds = dae_ecv_embeds.to_numpy()

dae_ff_embeds = pd.read_csv('extracted_features/forestfire_dae.csv', header=None)
#dae_ff_embeds = dae_ff_embeds.drop('Unnamed: 0', axis=1)
dae_ff_embeds = dae_ff_embeds.to_numpy()

dae_embeds = []
dae_embeds.append(dae_br_embeds)
dae_embeds.append(dae_ecv_embeds)
dae_embeds.append(dae_ff_embeds)

In [39]:
# Extract DCAE embeddings; not re-doing DCAE in this notebook
dcae_br_embeds = pd.read_csv('extracted_features/bank_reserves_dcae.csv', header=None)
#dcae_br_embeds = dcae_br_embeds.drop('Unnamed: 0', axis=1)
dcae_br_embeds = dcae_br_embeds.to_numpy()

dcae_ecv_embeds = pd.read_csv('extracted_features/epstein_dcae.csv', header=None)
#dcae_ecv_embeds = dcae_ecv_embeds.drop('Unnamed: 0', axis=1)
dcae_ecv_embeds = dcae_ecv_embeds.to_numpy()

dcae_ff_embeds = pd.read_csv('extracted_features/forestfire_dcae.csv', header=None)
#dcae_ff_embeds = dcae_ff_embeds.drop('Unnamed: 0', axis=1)
dcae_ff_embeds = dcae_ff_embeds.to_numpy()

dcae_embeds = []
dcae_embeds.append(dcae_br_embeds)
dcae_embeds.append(dcae_ecv_embeds)
dcae_embeds.append(dcae_ff_embeds)

In [40]:
# Extract DAEP embeddings; not re-doing DAEP in this notebook
daep_br_embeds = pd.read_csv('extracted_features/0_DAE_5_2_2048_linear.csv', header=None)
#daep_br_embeds = daep_br_embeds.drop('Unnamed: 0', axis=1)
daep_br_embeds = daep_br_embeds.to_numpy()

daep_ecv_embeds = pd.read_csv('extracted_features/1_DAE_5_3_2048_linear.csv', header=None)
#daep_ecv_embeds = daep_ecv_embeds.drop('Unnamed: 0', axis=1)
daep_ecv_embeds = daep_ecv_embeds.to_numpy()

daep_ff_embeds = pd.read_csv('extracted_features/2_DAE_3_5_2048_linear.csv', header=None)
#daep_ff_embeds = daep_ff_embeds.drop('Unnamed: 0', axis=1)
daep_ff_embeds = daep_ff_embeds.to_numpy()

daep_embeds = []
daep_embeds.append(daep_br_embeds)
daep_embeds.append(daep_ecv_embeds)
daep_embeds.append(daep_ff_embeds)

In [41]:
opt_clusters = []
opt_clusters.append(7)
opt_clusters.append(8)
opt_clusters.append(4)

# Cluster Raw Data

In [21]:
out_path = Path("raw_sil_scores.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            flat_list = [sample.flatten() for sample in raw_data[i]]
            raw_features = np.asarray(flat_list)

            km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
            raw_labels = km.fit(raw_features).labels_
            unique_labels = np.unique(raw_labels)

            if len(unique_labels) > 1:
                score = sil_score(raw_features, raw_labels, sil_sample)
                print(f"[Raw data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('raw', abm, k, raw_labels, json_path="raw_labels.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print("Error: Too few labels generated. Skipping ...")

C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\subprocess.py",

Silhouette Score: 0.6218
[Raw data,BR,3] 	 Silhouette Score: 0.6218
Silhouette Score: 0.5445
[Raw data,BR,4] 	 Silhouette Score: 0.5445
Silhouette Score: 0.4739
[Raw data,BR,5] 	 Silhouette Score: 0.4739
Silhouette Score: 0.3163
[Raw data,BR,6] 	 Silhouette Score: 0.3163
Silhouette Score: 0.2885
[Raw data,BR,7] 	 Silhouette Score: 0.2885
Silhouette Score: 0.2560
[Raw data,BR,8] 	 Silhouette Score: 0.2560
Silhouette Score: 0.2364
[Raw data,BR,9] 	 Silhouette Score: 0.2364
Silhouette Score: 0.2132
[Raw data,BR,10] 	 Silhouette Score: 0.2132
Silhouette Score: 0.6034
[Raw data,ECV,3] 	 Silhouette Score: 0.6034
Silhouette Score: 0.5946
[Raw data,ECV,4] 	 Silhouette Score: 0.5946
Silhouette Score: 0.5840
[Raw data,ECV,5] 	 Silhouette Score: 0.5840
Silhouette Score: 0.5898
[Raw data,ECV,6] 	 Silhouette Score: 0.5898
Silhouette Score: 0.5281
[Raw data,ECV,7] 	 Silhouette Score: 0.5281
Silhouette Score: 0.5251
[Raw data,ECV,8] 	 Silhouette Score: 0.5251
Silhouette Score: 0.5175
[Raw data,ECV,9]

# Kmean Cluster Reference Methods (PCA, DAE, DCAE)

In [22]:
out_path = Path("pca_sil_scores.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            pca_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
            pca_labels = pca_model.fit(pca_embeds[i]).labels_
            unique_labels = np.unique(pca_labels)

            if len(unique_labels) > 1:
                score = sil_score(pca_embeds[i], pca_labels, sil_sample)
                print(f"[PCA data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('pca', abm, k, pca_labels, json_path="pca_labels.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.7625
[PCA data,BR,3] 	 Silhouette Score: 0.7625
Silhouette Score: 0.6950
[PCA data,BR,4] 	 Silhouette Score: 0.6950
Silhouette Score: 0.6041
[PCA data,BR,5] 	 Silhouette Score: 0.6041
Silhouette Score: 0.5890
[PCA data,BR,6] 	 Silhouette Score: 0.5890
Silhouette Score: 0.5566
[PCA data,BR,7] 	 Silhouette Score: 0.5566
Silhouette Score: 0.5487
[PCA data,BR,8] 	 Silhouette Score: 0.5487
Silhouette Score: 0.5370
[PCA data,BR,9] 	 Silhouette Score: 0.5370
Silhouette Score: 0.5243
[PCA data,BR,10] 	 Silhouette Score: 0.5243
Silhouette Score: 0.6228
[PCA data,ECV,3] 	 Silhouette Score: 0.6228
Silhouette Score: 0.6210
[PCA data,ECV,4] 	 Silhouette Score: 0.6210
Silhouette Score: 0.6165
[PCA data,ECV,5] 	 Silhouette Score: 0.6165
Silhouette Score: 0.6207
[PCA data,ECV,6] 	 Silhouette Score: 0.6207
Silhouette Score: 0.5668
[PCA data,ECV,7] 	 Silhouette Score: 0.5668
Silhouette Score: 0.5704
[PCA data,ECV,8] 	 Silhouette Score: 0.5704
Silhouette Score: 0.5679
[PCA data,ECV,9]

In [23]:
out_path = Path("dae_sil_scores.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            dae_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
            dae_labels = dae_model.fit(dae_embeds[i]).labels_
            #print(dae_embeds[i].size)
            unique_labels = np.unique(dae_labels)

            if len(unique_labels) > 1:
                score = sil_score(dae_embeds[i], dae_labels, sil_sample)
                print(f"[DAE data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('dae', abm, k, dae_labels, json_path="dae_labels.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.7399
[DAE data,BR,3] 	 Silhouette Score: 0.7399
Silhouette Score: 0.7100
[DAE data,BR,4] 	 Silhouette Score: 0.7100
Silhouette Score: 0.6837
[DAE data,BR,5] 	 Silhouette Score: 0.6837
Silhouette Score: 0.6577
[DAE data,BR,6] 	 Silhouette Score: 0.6577
Silhouette Score: 0.5746
[DAE data,BR,7] 	 Silhouette Score: 0.5746
Silhouette Score: 0.5684
[DAE data,BR,8] 	 Silhouette Score: 0.5684
Silhouette Score: 0.5689
[DAE data,BR,9] 	 Silhouette Score: 0.5689
Silhouette Score: 0.5629
[DAE data,BR,10] 	 Silhouette Score: 0.5629
Silhouette Score: 0.5507
[DAE data,ECV,3] 	 Silhouette Score: 0.5507
Silhouette Score: 0.5703
[DAE data,ECV,4] 	 Silhouette Score: 0.5703
Silhouette Score: 0.5592
[DAE data,ECV,5] 	 Silhouette Score: 0.5592
Silhouette Score: 0.5694
[DAE data,ECV,6] 	 Silhouette Score: 0.5694
Silhouette Score: 0.5642
[DAE data,ECV,7] 	 Silhouette Score: 0.5642
Silhouette Score: 0.5404
[DAE data,ECV,8] 	 Silhouette Score: 0.5404
Silhouette Score: 0.5494
[DAE data,ECV,9]

In [24]:
out_path = Path("dcae_sil_scores.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            dcae_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
            dcae_labels = dcae_model.fit(dcae_embeds[i]).labels_
            unique_labels = np.unique(dcae_labels)

            if len(unique_labels) > 1:
                score = sil_score(dcae_embeds[i], dcae_labels, sil_sample)
                print(f"[dcae data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('dcae', abm, k, dcae_labels, json_path="dcae_labels.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.5995
[dcae data,BR,3] 	 Silhouette Score: 0.5995
Silhouette Score: 0.5234
[dcae data,BR,4] 	 Silhouette Score: 0.5234
Silhouette Score: 0.4282
[dcae data,BR,5] 	 Silhouette Score: 0.4282
Silhouette Score: 0.4213
[dcae data,BR,6] 	 Silhouette Score: 0.4213
Silhouette Score: 0.4094
[dcae data,BR,7] 	 Silhouette Score: 0.4094
Silhouette Score: 0.3881
[dcae data,BR,8] 	 Silhouette Score: 0.3881
Silhouette Score: 0.3784
[dcae data,BR,9] 	 Silhouette Score: 0.3784
Silhouette Score: 0.3674
[dcae data,BR,10] 	 Silhouette Score: 0.3674
Silhouette Score: 0.5631
[dcae data,ECV,3] 	 Silhouette Score: 0.5631
Silhouette Score: 0.5706
[dcae data,ECV,4] 	 Silhouette Score: 0.5706
Silhouette Score: 0.5571
[dcae data,ECV,5] 	 Silhouette Score: 0.5571
Silhouette Score: 0.5589
[dcae data,ECV,6] 	 Silhouette Score: 0.5589
Silhouette Score: 0.5122
[dcae data,ECV,7] 	 Silhouette Score: 0.5122
Silhouette Score: 0.5273
[dcae data,ECV,8] 	 Silhouette Score: 0.5273
Silhouette Score: 0.5324
[d

# Kmeans DAE Plus

In [46]:
out_path = Path("daep_sil_scores.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            daep_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
            daep_labels = daep_model.fit(daep_embeds[i]).labels_
            unique_labels = np.unique(daep_labels)

            if len(unique_labels) > 1:
                score = sil_score(daep_embeds[i], daep_labels, sil_sample)
                print(f"[daep data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('daep', abm, k, daep_labels, json_path="daep_labels.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.7621
[daep data,BR,3] 	 Silhouette Score: 0.7621
Silhouette Score: 0.7391
[daep data,BR,4] 	 Silhouette Score: 0.7391
Silhouette Score: 0.7194
[daep data,BR,5] 	 Silhouette Score: 0.7194
Silhouette Score: 0.6966
[daep data,BR,6] 	 Silhouette Score: 0.6966
Silhouette Score: 0.6695
[daep data,BR,7] 	 Silhouette Score: 0.6695
Silhouette Score: 0.6283
[daep data,BR,8] 	 Silhouette Score: 0.6283
Silhouette Score: 0.5768
[daep data,BR,9] 	 Silhouette Score: 0.5768
Silhouette Score: 0.5562
[daep data,BR,10] 	 Silhouette Score: 0.5562
Silhouette Score: 0.8935
[daep data,ECV,3] 	 Silhouette Score: 0.8935
Silhouette Score: 0.8717
[daep data,ECV,4] 	 Silhouette Score: 0.8717
Silhouette Score: 0.8308
[daep data,ECV,5] 	 Silhouette Score: 0.8308
Silhouette Score: 0.8318
[daep data,ECV,6] 	 Silhouette Score: 0.8318
Silhouette Score: 0.8217
[daep data,ECV,7] 	 Silhouette Score: 0.8217
Silhouette Score: 0.8213
[daep data,ECV,8] 	 Silhouette Score: 0.8213
Silhouette Score: 0.6675
[d

# Agglom-KNN DAE Plus

In [44]:
out_path = Path("daep_sil_scores_agglom_knn.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            daep_labels = agglomerative(daep_embeds[i], k, "euclidean", "knn")
            unique_labels = np.unique(daep_labels)

            if len(unique_labels) > 1:
                score = sil_score(daep_embeds[i], daep_labels, sil_sample)
                print(f"[daep data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('daep', abm, k, daep_labels, json_path="daep_labels_agglom_knn.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.7507
[daep data,BR,3] 	 Silhouette Score: 0.7507
Silhouette Score: 0.6895
[daep data,BR,4] 	 Silhouette Score: 0.6895
Silhouette Score: 0.6860
[daep data,BR,5] 	 Silhouette Score: 0.6860
Silhouette Score: 0.6851
[daep data,BR,6] 	 Silhouette Score: 0.6851
Silhouette Score: 0.6635
[daep data,BR,7] 	 Silhouette Score: 0.6635
Silhouette Score: 0.5879
[daep data,BR,8] 	 Silhouette Score: 0.5879
Silhouette Score: 0.5861
[daep data,BR,9] 	 Silhouette Score: 0.5861
Silhouette Score: 0.5459
[daep data,BR,10] 	 Silhouette Score: 0.5459
Silhouette Score: 0.8955
[daep data,ECV,3] 	 Silhouette Score: 0.8955
Silhouette Score: 0.8872
[daep data,ECV,4] 	 Silhouette Score: 0.8872
Silhouette Score: 0.8249
[daep data,ECV,5] 	 Silhouette Score: 0.8249
Silhouette Score: 0.8245
[daep data,ECV,6] 	 Silhouette Score: 0.8245
Silhouette Score: 0.8065
[daep data,ECV,7] 	 Silhouette Score: 0.8065
Silhouette Score: 0.7759
[daep data,ECV,8] 	 Silhouette Score: 0.7759
Silhouette Score: 0.7754
[d

# Agglom-centroids DAE Plus

In [27]:
out_path = Path("daep_sil_scores_agglom_centroids.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            daep_labels = agglom_dtw(daep_embeds[i], k, 1)
            unique_labels = np.unique(daep_labels)

            if len(unique_labels) > 1:
                score = sil_score(daep_embeds[i], daep_labels, sil_sample)
                print(f"[daep data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('daep', abm, k, daep_labels, json_path="daep_labels_agglom_centroid.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.7646
[daep data,BR,3] 	 Silhouette Score: 0.7646
Silhouette Score: 0.7324
[daep data,BR,4] 	 Silhouette Score: 0.7324
Silhouette Score: 0.7206
[daep data,BR,5] 	 Silhouette Score: 0.7206
Silhouette Score: 0.6903
[daep data,BR,6] 	 Silhouette Score: 0.6903
Silhouette Score: 0.6789
[daep data,BR,7] 	 Silhouette Score: 0.6789
Silhouette Score: 0.6711
[daep data,BR,8] 	 Silhouette Score: 0.6711
Silhouette Score: 0.6538
[daep data,BR,9] 	 Silhouette Score: 0.6538
Silhouette Score: 0.6339
[daep data,BR,10] 	 Silhouette Score: 0.6339
Silhouette Score: 0.8910
[daep data,ECV,3] 	 Silhouette Score: 0.8910
Silhouette Score: 0.8737
[daep data,ECV,4] 	 Silhouette Score: 0.8737
Silhouette Score: 0.8737
[daep data,ECV,5] 	 Silhouette Score: 0.8737
Silhouette Score: 0.8598
[daep data,ECV,6] 	 Silhouette Score: 0.8598
Silhouette Score: 0.8604
[daep data,ECV,7] 	 Silhouette Score: 0.8604
Silhouette Score: 0.8592
[daep data,ECV,8] 	 Silhouette Score: 0.8592
Silhouette Score: 0.8091
[d

# Spectral DAE Plus

In [28]:
out_path = Path("daep_sil_scores_spectral.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            # Train spectral clustering on a subset for computational efficiency
            spectral_model = SpectralClustering(
                n_clusters=k,
                assign_labels="discretize",
                affinity="nearest_neighbors",
                random_state=0
            )

            subset = daep_embeds[i][1:10000]
            spectral_labs = spectral_model.fit(subset).labels_

            # Extend labels to full dataset via k-NN propagation
            neigh = KNeighborsClassifier(n_neighbors=13)
            neigh.fit(subset, spectral_labs)
            daep_labels = neigh.predict(daep_embeds[i])

            unique_labels = np.unique(daep_labels)

            if len(unique_labels) > 1:
                score = sil_score(daep_embeds[i], daep_labels, sil_sample)
                print(f"[daep data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('daep', abm, k, spectral_labs, json_path="daep_labels_spectral.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.6913
[daep data,BR,3] 	 Silhouette Score: 0.6913
Silhouette Score: 0.6875
[daep data,BR,4] 	 Silhouette Score: 0.6875
Silhouette Score: 0.6418
[daep data,BR,5] 	 Silhouette Score: 0.6418
Silhouette Score: 0.5952
[daep data,BR,6] 	 Silhouette Score: 0.5952
Silhouette Score: 0.5302
[daep data,BR,7] 	 Silhouette Score: 0.5302
Silhouette Score: 0.4111
[daep data,BR,8] 	 Silhouette Score: 0.4111
Silhouette Score: 0.4202
[daep data,BR,9] 	 Silhouette Score: 0.4202
Silhouette Score: 0.4293
[daep data,BR,10] 	 Silhouette Score: 0.4293
Silhouette Score: 0.0865
[daep data,ECV,3] 	 Silhouette Score: 0.0865
Silhouette Score: 0.1058
[daep data,ECV,4] 	 Silhouette Score: 0.1058
Silhouette Score: 0.2874
[daep data,ECV,5] 	 Silhouette Score: 0.2874
Silhouette Score: 0.3123
[daep data,ECV,6] 	 Silhouette Score: 0.3123
Silhouette Score: 0.3777
[daep data,ECV,7] 	 Silhouette Score: 0.3777
Silhouette Score: 0.3763
[daep data,ECV,8] 	 Silhouette Score: 0.3763
Silhouette Score: 0.3959
[d

C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: 0.3496
[daep data,FF,3] 	 Silhouette Score: 0.3496


C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: -0.0008
[daep data,FF,4] 	 Silhouette Score: -0.0008


...

Silhouette Score: -0.0870
[daep data,FF,8] 	 Silhouette Score: -0.0870


C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: 0.3050
[daep data,FF,9] 	 Silhouette Score: 0.3050


C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: 0.0290
[daep data,FF,10] 	 Silhouette Score: 0.0290


In [43]:
out_path = Path("daep_sil_scores_agglom_cent_euc.csv")
write_header = not out_path.exists() or out_path.stat().st_size == 0

with out_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["abm", "k", "score", "labels"])  # header once

    for i, abm in enumerate(ABMs):
        for k in range(3, 11):
            daep_labels = agglomerative(daep_embeds[i], k, "euclidean", "centroid")
            unique_labels = np.unique(daep_labels)

            if len(unique_labels) > 1:
                score = sil_score(daep_embeds[i], daep_labels, sil_sample)
                print(f"[daep data,{abm},{k}] \t Silhouette Score: {score:.4f}")
                save_labels_ndjson('daep', abm, k, daep_labels, json_path="daep_labels_agglom_centroid_euc.ndjson")
                writer.writerow([abm, k, f"{score:.4f}"])
            else:
                print(f"Error: Too few labels generated for {abm} (k={k}). Skipping ...")

Silhouette Score: 0.7643
[daep data,BR,3] 	 Silhouette Score: 0.7643
Silhouette Score: 0.7323
[daep data,BR,4] 	 Silhouette Score: 0.7323
Silhouette Score: 0.7198
[daep data,BR,5] 	 Silhouette Score: 0.7198
Silhouette Score: 0.6975
[daep data,BR,6] 	 Silhouette Score: 0.6975
Silhouette Score: 0.6731
[daep data,BR,7] 	 Silhouette Score: 0.6731
Silhouette Score: 0.5982
[daep data,BR,8] 	 Silhouette Score: 0.5982
Silhouette Score: 0.5968
[daep data,BR,9] 	 Silhouette Score: 0.5968
Silhouette Score: 0.5676
[daep data,BR,10] 	 Silhouette Score: 0.5676
Silhouette Score: 0.8978
[daep data,ECV,3] 	 Silhouette Score: 0.8978
Silhouette Score: 0.8872
[daep data,ECV,4] 	 Silhouette Score: 0.8872
Silhouette Score: 0.8366
[daep data,ECV,5] 	 Silhouette Score: 0.8366
Silhouette Score: 0.8343
[daep data,ECV,6] 	 Silhouette Score: 0.8343
Silhouette Score: 0.8162
[daep data,ECV,7] 	 Silhouette Score: 0.8162
Silhouette Score: 0.7719
[daep data,ECV,8] 	 Silhouette Score: 0.7719
Silhouette Score: 0.7743
[d